# Small-task exclusion ablation

Four benchmark tasks have fewer than 500 training examples. This notebook
asks whether the model rankings depend on them. Each is removed from its
category **one at a time**, and the category score of every model is
recomputed and re-ranked.

| Excluded task | Category | $K$ |
|---|---|---|
| `open_kidney_capsule_segmentation` | Anatomical segmentation | 6 → 5 |
| `open_kidney_regions_segmentation` | Anatomical segmentation | 6 → 5 |
| `masld_classification` | Pathological classification | 8 → 7 |
| `covid_blues_classification` | Pathological classification | 8 → 7 |

The two other categories hold none of these tasks, so they are unaffected
and left out. Only full fine-tuning is considered.

**Method.** The category score and its standard error are exactly those of
`simplified_protocol.ipynb` (`docs/evaluation_protocol.md`, section 3.5),
computed over the $K - 1$ remaining tasks:

$$\bar{Y} = \frac{1}{K}\sum_{k=1}^{K}\bar{Y}_k,\qquad
\mathrm{SE}(\bar{Y}) = \sqrt{\frac{\sigma_{\mathrm{run}}^{2}}{S} + \sigma_{\mathrm{boot}}^{2}}.$$

Nothing is retrained or re-scored. The notebook reads the per-row
contributions that `simplified_protocol.ipynb` cached and replays its frozen
bootstrap draws, so the baseline column reproduces
`simplified_protocol_category_intervals.csv` exactly, which is checked below.

In [ ]:
import html
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from IPython.display import HTML, display
from scipy.stats import kendalltau, spearmanr

from ultrabench2.evaluate.tasks import CATEGORIES, TaskSpec, get_task

RESAMPLE_SEED = 20260906
"""The protocol's frozen seed. Must match `simplified_protocol.ipynb`."""

N_RESAMPLES = 2000
"""Number of test-set resamples drawn per source dataset (B in the formula)."""

PATIENT_COLUMN = "patient"
"""The dataset column holding a per-image patient identifier, where present."""

CACHE_DIR = Path("simplified_protocol_cache_20260908_163207")
"""The per-row contributions cached by `simplified_protocol.ipynb`."""

TASK_TERMS_CSV = Path("simplified_protocol_task_terms.csv")
CATEGORY_INTERVALS_CSV = Path("simplified_protocol_category_intervals.csv")
"""Outputs of `simplified_protocol.ipynb`, used only as validation targets."""

PARADIGM = "full_finetuning"

EXCLUDED_TASKS = [
    "open_kidney_capsule_segmentation",
    "open_kidney_regions_segmentation",
    "masld_classification",
    "covid_blues_classification",
]
"""The tasks with fewer than 500 training examples, removed one at a time."""

MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "tinyusfm": "TinyUSFM",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}

TASK_DISPLAY_NAMES = {
    "open_kidney_capsule_segmentation": "Open Kidney Capsule",
    "open_kidney_regions_segmentation": "Open Kidney Regions",
    "masld_classification": "MASLD",
    "covid_blues_classification": "COVID-BLUES",
}

TASK_CATEGORY = {
    task: name for name, category in CATEGORIES.items() for task in category.tasks
}
TASK_NAMES = sorted(TASK_CATEGORY)

for _task in EXCLUDED_TASKS:
    if _task not in TASK_CATEGORY:
        raise ValueError(f"'{_task}' is not a benchmark task.")

AFFECTED_CATEGORIES = sorted({TASK_CATEGORY[task] for task in EXCLUDED_TASKS})
NEEDED_TASKS = sorted(
    task for name in AFFECTED_CATEGORIES for task in CATEGORIES[name].tasks
)

pd.DataFrame(
    {
        "excluded_task": EXCLUDED_TASKS,
        "category": [TASK_CATEGORY[task] for task in EXCLUDED_TASKS],
        "k_tasks": [
            f"{len(CATEGORIES[TASK_CATEGORY[task]].tasks)} -> "
            f"{len(CATEGORIES[TASK_CATEGORY[task]].tasks) - 1}"
            for task in EXCLUDED_TASKS
        ],
    }
)

## The protocol's machinery

The functions below are **copies** of those in `simplified_protocol.ipynb`
(the frozen resample, the row and unit bookkeeping, and the vectorised
metrics). They are copied rather than imported so that this ablation changes
no other file. Two differences, both read-only:

* `load_patient_map` reads the patient maps the protocol cached and raises if
  one is missing, instead of falling back to downloading the dataset.
* `load_task_cache` loads the cached contributions and raises if one is
  missing, instead of rebuilding them from the run directories.

**The unit lists span all 21 tasks.** A dataset's bootstrap draw depends on
its full set of test units, and several datasets (AUL, BUS-BRA, MMOTU, TN3K,
CAMUS) have tasks in the unaffected categories too. Building the units from
the affected categories alone would silently change the draws.

In [ ]:
def resample_indices(
    unit_ids: np.ndarray, dataset: str, n_resamples: int = N_RESAMPLES
) -> np.ndarray:
    """Draw the frozen bootstrap resamples of a source dataset's test units.

    Copied from `simplified_protocol.ipynb`.

    Args:
        unit_ids: The unit IDs of the dataset's test split, in any order and
            with any repeats.
        dataset: The name of the source dataset, mixed into the seed so that
            different datasets draw independently.
        n_resamples: The number of resamples to draw.

    Returns:
        An array of shape (n_resamples, n_units) holding the unit IDs drawn
        for each resample.
    """
    units = np.unique(unit_ids)  # sorted, canonical
    seed = [RESAMPLE_SEED, zlib.crc32(dataset.encode())]  # frozen + dataset identity
    rng = np.random.default_rng(seed)  # PCG64: stream is version-stable

    return units[rng.integers(0, len(units), (n_resamples, len(units)))]


def resample_positions(
    units: np.ndarray, dataset: str, n_resamples: int = N_RESAMPLES
) -> np.ndarray:
    """Express the frozen resamples as positions into the sorted unit array.

    Copied from `simplified_protocol.ipynb`.

    Args:
        units: The sorted unique unit IDs of a dataset's test split.
        dataset: The name of the source dataset.
        n_resamples: The number of resamples to draw.

    Returns:
        An array of shape (n_resamples, n_units) of positions into `units`.

    Raises:
        ValueError: If the positions do not reproduce `resample_indices`.
    """
    draws = resample_indices(units, dataset, n_resamples)
    positions = np.searchsorted(units, draws)

    if not np.array_equal(units[positions], draws):
        raise ValueError("Resample positions do not reproduce the drawn unit IDs.")

    return positions

In [ ]:
# Copied from `simplified_protocol.ipynb`, which documents why the mapping is
# written out by hand.
GROUND_TRUTH_FILES = {
    "aul_liver_segmentation": "test.256-ground-truth-segmentation-liver.csv",
    "aul_mass_classification": "test.256-ground-truth-classification.csv",
    "aul_mass_segmentation": "test.256-ground-truth-segmentation-mass.csv",
    "busbra_histology_classification": (
        "test.256-ground-truth-classification-histology.csv"
    ),
    "busbra_pathology_classification": (
        "test.256-ground-truth-classification-pathology.csv"
    ),
    "busbra_segmentation": "test.256-ground-truth-segmentation.csv",
    "camus_classification": "classification-test.256-ground-truth-classification.csv",
    "camus_segmentation": "segmentation-test.256-ground-truth-segmentation.csv",
    "covid_blues_classification": "test.256-ground-truth-classification.csv",
    "fetal_planes_brain_plane_classification": (
        "fetal-brain-test.256-ground-truth-classification.csv"
    ),
    "fetal_planes_plane_classification": (
        "fetal-test.256-ground-truth-classification.csv"
    ),
    "gbcu_classification": "test.256-ground-truth-classification.csv",
    "masld_classification": "test.256-ground-truth-classification.csv",
    "mmotu_classification": "test.256-ground-truth-classification.csv",
    "mmotu_segmentation": "test.256-ground-truth-segmentation.csv",
    "open_kidney_capsule_segmentation": (
        "test.256-ground-truth-segmentation-capsule.csv"
    ),
    "open_kidney_regions_segmentation": (
        "test.256-ground-truth-segmentation-regions.csv"
    ),
    "psfhs_segmentation": "test.256-ground-truth-segmentation.csv",
    "stmus_segmentation": "test.256-ground-truth-segmentation.csv",
    "tn3k_classification": "test.256-ground-truth-classification.csv",
    "tn3k_segmentation": "test.256-ground-truth-segmentation.csv",
}

if set(GROUND_TRUTH_FILES) != set(TASK_NAMES):
    raise ValueError("GROUND_TRUTH_FILES does not name every task exactly once.")


def ground_truth_csv(task: str) -> Path:
    """Download the ground-truth CSV published with a task's dataset.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.

    Returns:
        The path of the CSV in the local Hugging Face cache.
    """
    spec = get_task(task)

    return Path(
        hf_hub_download(
            spec.dataset,
            GROUND_TRUTH_FILES[task],
            repo_type="dataset",
            revision=spec.revision,
        )
    )


def load_ground_truth(task: str) -> pd.DataFrame:
    """Read the ground-truth CSV published with a task's dataset.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.

    Returns:
        The ground truth, with `encoded_pixels` kept as a string so that an
        empty mask does not become NaN.
    """
    return pd.read_csv(ground_truth_csv(task), converters={"encoded_pixels": str})


def row_column(task: str) -> str:
    """Name the ground-truth column identifying a task's scored rows.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.

    Returns:
        `sequence_id` for sequence-level tasks, `image_id` otherwise.
    """
    return "sequence_id" if get_task(task).sequence_by else "image_id"


def task_rows(task: str) -> np.ndarray:
    """List the sorted unique scored rows of a task's test split.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.

    Returns:
        The sorted unique row IDs, as strings.
    """
    return np.unique(load_ground_truth(task)[row_column(task)].astype(str).to_numpy())


def load_patient_map(spec: TaskSpec) -> dict[str, str] | None:
    """Read the cached image-to-patient map of a test split, if it has one.

    Adapted from `simplified_protocol.ipynb`: the map is only ever read from
    the protocol's cache, never built.

    Args:
        spec: The task whose source split to read.

    Returns:
        A mapping of image ID to patient ID, or None if the dataset provides
        no patient identifier.

    Raises:
        FileNotFoundError: If `simplified_protocol.ipynb` has not cached the
            map.
    """
    subset = spec.subset or "default"
    path = CACHE_DIR / f"patients_{spec.dataset.replace('/', '-')}_{subset}.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing; run simplified_protocol.ipynb.")

    table = pd.read_csv(path, dtype=str)
    if table.empty:
        return None

    return dict(zip(table["image_id"], table[PATIENT_COLUMN]))


def task_units(task: str) -> tuple[np.ndarray, str]:
    """Give the resampling unit of each scored row of a task.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.

    Returns:
        The unit ID of each row in sorted-row order, and whether the units are
        patients or images.

    Raises:
        ValueError: If a sequence spans more than one patient.
    """
    spec = get_task(task)
    rows = ROWS[task]
    patients = load_patient_map(spec)

    if patients is None:
        return rows.copy(), "image"

    if spec.sequence_by is None:
        return np.array([patients[row] for row in rows]), "patient"

    ground_truth = load_ground_truth(task)
    by_sequence = (
        ground_truth["image_id"]
        .astype(str)
        .map(patients)
        .groupby(ground_truth["sequence_id"].astype(str))
        .agg(lambda values: set(values))
    )
    split_sequences = by_sequence[by_sequence.map(len) != 1]
    if len(split_sequences) > 0:
        raise ValueError(
            f"Task '{task}' has {len(split_sequences)} sequence(s) spanning more "
            f"than one patient, starting with '{split_sequences.index[0]}'."
        )

    sequence_patient = by_sequence.map(lambda values: next(iter(values)))

    return np.array([sequence_patient[row] for row in rows]), "patient"


# Built over every task, so each dataset's unit list -- and so its draw -- is
# the protocol's.
ROWS = {task: task_rows(task) for task in TASK_NAMES}
ROW_UNITS = {task: task_units(task) for task in TASK_NAMES}

DATASET_TASKS: dict[str, list[str]] = {}
for task in TASK_NAMES:
    DATASET_TASKS.setdefault(get_task(task).dataset, []).append(task)

DATASET_UNITS = {
    dataset: np.unique(np.concatenate([ROW_UNITS[task][0] for task in tasks]))
    for dataset, tasks in DATASET_TASKS.items()
}
UNIT_INDEX = {
    task: np.searchsorted(DATASET_UNITS[get_task(task).dataset], ROW_UNITS[task][0])
    for task in TASK_NAMES
}

for task in TASK_NAMES:
    units = DATASET_UNITS[get_task(task).dataset]
    if not np.array_equal(units[UNIT_INDEX[task]], ROW_UNITS[task][0]):
        raise ValueError(f"Task '{task}' has rows outside its dataset's unit list.")

print(f"{len(TASK_NAMES)} tasks over {len(DATASET_UNITS)} source datasets.")

In [ ]:
def load_task_cache(task: str) -> dict[str, np.ndarray]:
    """Load a task's per-row contributions cached by the protocol notebook.

    Args:
        task: The name of the task.

    Returns:
        A mapping of cache key to array.

    Raises:
        FileNotFoundError: If `simplified_protocol.ipynb` has not cached the
            task.
    """
    path = CACHE_DIR / f"{task}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing; run simplified_protocol.ipynb.")

    with np.load(path, allow_pickle=False) as archive:
        return {key: archive[key] for key in archive.files}


def multiplicity_matrix(dataset: str) -> np.ndarray:
    """Count how many times each of a dataset's units is drawn per replication.

    Copied from `simplified_protocol.ipynb`.

    Args:
        dataset: The name of the source dataset.

    Returns:
        A (N_RESAMPLES, n_units) matrix of draw counts.
    """
    units = DATASET_UNITS[dataset]
    positions = resample_positions(units, dataset)
    n_units = len(units)

    flat = positions + np.arange(N_RESAMPLES)[:, None] * n_units
    counts = np.bincount(flat.ravel(), minlength=N_RESAMPLES * n_units)

    return counts.reshape(N_RESAMPLES, n_units).astype(np.float32)


def macro_f1_from_confusion(confusion: np.ndarray) -> np.ndarray:
    """Compute macro F1 from a stack of confusion matrices.

    Copied from `simplified_protocol.ipynb`.

    Args:
        confusion: Confusion matrices of shape (n, n_classes, n_classes),
            indexed as [replication, true, predicted].

    Returns:
        The macro F1 of each confusion matrix.
    """
    true_positives = np.einsum("bii->bi", confusion)
    # 2*TP + FP + FN reduces to the row plus column sums of the class.
    denominator = confusion.sum(axis=2) + confusion.sum(axis=1)
    f1 = np.divide(
        2.0 * true_positives,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0,
    )

    return f1.sum(axis=1) / (denominator > 0).sum(axis=1)


def unit_cell_counts(task: str, key: str) -> np.ndarray:
    """Count each unit's (true, predicted) confusion cells for one run.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.
        key: The run's cache key.

    Returns:
        A (n_dataset_units, n_classes ** 2) matrix of counts.
    """
    n_classes = get_task(task).num_classes
    cache = CONTRIBUTIONS[task]
    labels_gt = cache["label_gt"]
    labels_pred = cache[f"pred|{key}"]
    n_units = len(DATASET_UNITS[get_task(task).dataset])

    counts = np.zeros((n_units, n_classes * n_classes), dtype=np.float32)
    np.add.at(counts, (UNIT_INDEX[task], labels_gt * n_classes + labels_pred), 1.0)

    return counts


def unit_dice_sums(task: str, key: str) -> tuple[np.ndarray, np.ndarray]:
    """Sum one run's per-image Dice scores within each unit.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.
        key: The run's cache key.

    Returns:
        The per-unit sum of defined Dice scores and the per-unit count of
        defined images.
    """
    values = CONTRIBUTIONS[task][key]
    defined = np.isfinite(values).astype(np.float64)
    n_units = len(DATASET_UNITS[get_task(task).dataset])

    numerator = np.bincount(
        UNIT_INDEX[task], weights=np.nan_to_num(values) * defined, minlength=n_units
    )
    denominator = np.bincount(UNIT_INDEX[task], weights=defined, minlength=n_units)

    return numerator.astype(np.float32), denominator.astype(np.float32)


def resampled_scores(task: str, key: str, multiplicity: np.ndarray) -> np.ndarray:
    """Recompute a run's task metric on every replication.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.
        key: The run's cache key.
        multiplicity: The source dataset's multiplicity matrix.

    Returns:
        The metric of the run on each of the N_RESAMPLES replications.
    """
    if get_task(task).task_type == "segmentation":
        numerator, denominator = unit_dice_sums(task, key)

        return (multiplicity @ numerator) / (multiplicity @ denominator)

    n_classes = get_task(task).num_classes
    confusion = (multiplicity @ unit_cell_counts(task, key)).reshape(
        -1, n_classes, n_classes
    )

    return macro_f1_from_confusion(confusion)


def point_score(task: str, key: str) -> float:
    """Compute a run's task metric on the test set itself.

    Copied from `simplified_protocol.ipynb`.

    Args:
        task: The name of the task.
        key: The run's cache key.

    Returns:
        The metric of the run, unresampled.
    """
    if get_task(task).task_type == "segmentation":
        return float(np.nanmean(CONTRIBUTIONS[task][key]))

    n_classes = get_task(task).num_classes
    confusion = unit_cell_counts(task, key).sum(axis=0, dtype=np.float64)

    return float(macro_f1_from_confusion(confusion.reshape(1, n_classes, n_classes))[0])


CONTRIBUTIONS = {task: load_task_cache(task) for task in NEEDED_TASKS}

## Per-task scores

Every run of every task in the two affected categories is scored on the test
set and on each of the $B$ replications. The replication scores are then
averaged over the runs. This is the protocol's loop, restricted to full
fine-tuning. The run keys come from the cache itself, so the run directories
are not needed.

In [ ]:
def cached_runs(task: str) -> dict[str, dict[int, str]]:
    """List the cached full fine-tuning runs of a task.

    Args:
        task: The name of the task.

    Returns:
        A mapping of model to {seed: cache key}.
    """
    runs: dict[str, dict[int, str]] = {}
    for name in CONTRIBUTIONS[task]:
        key = name.removeprefix("pred|")
        if key == "label_gt":
            continue
        model, paradigm, seed = key.split("|")
        if paradigm == PARADIGM:
            runs.setdefault(model, {})[int(seed)] = key

    return runs


run_scores: dict[tuple[str, str], np.ndarray] = {}
replication_scores: dict[tuple[str, str], np.ndarray] = {}
seeds_seen = set()

for dataset, dataset_tasks in DATASET_TASKS.items():
    needed = [task for task in dataset_tasks if task in NEEDED_TASKS]
    if not needed:
        continue
    multiplicity = multiplicity_matrix(dataset)

    for task in needed:
        for model, by_seed in cached_runs(task).items():
            keys = [by_seed[seed] for seed in sorted(by_seed)]
            seeds_seen.add(tuple(sorted(by_seed)))
            run_scores[(task, model)] = np.array(
                [point_score(task, key) for key in keys]
            )
            replication_scores[(task, model)] = np.stack(
                [resampled_scores(task, key, multiplicity) for key in keys]
            ).mean(axis=0)

    print(f"  {dataset} done ({len(needed)} task(s))")

if len(seeds_seen) != 1:
    raise ValueError(f"Runs do not share one set of seeds: {sorted(seeds_seen)}")
SEEDS = list(next(iter(seeds_seen)))
MODELS = sorted({model for _, model in run_scores})
print(f"{len(MODELS)} models x {len(NEEDED_TASKS)} tasks x {len(SEEDS)} runs {SEEDS}")

## Category scores with one task excluded

`category_score` is the protocol's `category_intervals` formula for a single
model, over the category's tasks minus an optional excluded one. Both
variance terms are formed on the category score, so tasks that share a test
set stay correlated.

The ranks use the descending mean within each (category, scenario), with
ties sharing the best rank.

In [ ]:
BASELINE = "baseline"


def category_score(category: str, model: str, excluded: str | None = None) -> dict:
    """Compute a model's category score and SE, optionally without one task.

    Args:
        category: The name of the category.
        model: The internal model key.
        excluded: The task to drop from the category, or None for all tasks.

    Returns:
        The category mean, its standard error, the two variance terms and the
        number of tasks averaged.

    Raises:
        ValueError: If the excluded task is not in the category, or the model
            does not cover every remaining task.
    """
    tasks = list(CATEGORIES[category].tasks)
    if excluded is not None:
        if excluded not in tasks:
            raise ValueError(f"'{excluded}' is not a task of '{category}'.")
        tasks.remove(excluded)

    missing = [task for task in tasks if (task, model) not in run_scores]
    if missing:
        raise ValueError(f"{model} is missing {missing} of '{category}'.")

    per_run = np.stack([run_scores[(task, model)] for task in tasks])  # K x S
    per_replication = np.stack(
        [replication_scores[(task, model)] for task in tasks]
    )  # K x B
    n_runs = per_run.shape[1]

    sigma_run = per_run.mean(axis=0).std(ddof=1)
    sigma_boot = per_replication.mean(axis=0).std(ddof=1)

    return {
        "mean": per_run.mean(),
        "se": np.sqrt(sigma_run**2 / n_runs + sigma_boot**2),
        "sigma_run": sigma_run,
        "sigma_boot": sigma_boot,
        "k_tasks": len(tasks),
    }


scenarios = [(category, None) for category in AFFECTED_CATEGORIES] + [
    (TASK_CATEGORY[task], task) for task in EXCLUDED_TASKS
]

ablation = pd.DataFrame(
    [
        {
            "category": category,
            "display_name": CATEGORIES[category].display_name,
            "scenario": BASELINE if excluded is None else f"without_{excluded}",
            "excluded_task": excluded,
            "model": model,
            **category_score(category, model, excluded),
        }
        for category, excluded in scenarios
        for model in MODELS
    ]
)
ablation["rank"] = (
    ablation.groupby(["category", "scenario"])["mean"]
    .rank(ascending=False, method="min")
    .astype(int)
)

baseline = ablation[ablation["scenario"] == BASELINE].set_index(["category", "model"])
ablation["delta_mean"] = (
    ablation["mean"].to_numpy()
    - baseline.loc[
        list(zip(ablation["category"], ablation["model"])), "mean"
    ].to_numpy()
)
# Positive means the model moved up once the task was dropped.
ablation["delta_rank"] = (
    baseline.loc[list(zip(ablation["category"], ablation["model"])), "rank"].to_numpy()
    - ablation["rank"].to_numpy()
)

ablation.to_csv("small_task_exclusion_ablation.csv", index=False)
ablation.head()

## Validation

The per-task means must match `simplified_protocol_task_terms.csv`, and the
baseline must reproduce `simplified_protocol_category_intervals.csv`. The
means must match to floating-point precision. The SE terms come from
float32 multiplicity products, so they must match to $10^{-6}$. If both
hold, the draws and the formula are the protocol's own.

In [ ]:
task_terms = pd.read_csv(TASK_TERMS_CSV)
task_terms = task_terms[
    (task_terms["paradigm"] == PARADIGM) & task_terms["task"].isin(NEEDED_TASKS)
].set_index(["task", "model"])

task_gaps = [
    abs(run_scores[key].mean() - task_terms.loc[key, "mean"]) for key in run_scores
]
if len(task_gaps) != len(task_terms):
    raise ValueError("The cache and the task-terms CSV cover different runs.")

reference = pd.read_csv(CATEGORY_INTERVALS_CSV)
reference = reference[reference["paradigm"] == PARADIGM].set_index(
    ["category", "model"]
)
check = baseline.join(reference, rsuffix="_ref", how="inner")
if len(check) != len(baseline):
    raise ValueError("The baseline and the category-intervals CSV do not align.")

gaps = {
    column: (check[column] - check[f"{column}_ref"]).abs().max()
    for column in ("mean", "se", "sigma_run", "sigma_boot")
}
print(f"max |task mean - task_terms.csv|   = {max(task_gaps):.3e}")
for column, gap in gaps.items():
    print(f"max |baseline {column:<10} - intervals.csv| = {gap:.3e}")

assert max(task_gaps) < 1e-9
assert gaps["mean"] < 1e-9
assert max(gaps["se"], gaps["sigma_run"], gaps["sigma_boot"]) < 1e-6
assert (check["k_tasks"] == check["k_tasks_ref"]).all()
assert (
    ablation.loc[ablation["excluded_task"].notna(), "k_tasks"]
    == ablation.loc[ablation["excluded_task"].notna(), "category"].map(
        lambda name: len(CATEGORIES[name].tasks) - 1
    )
).all()
print("Baseline reproduces simplified_protocol.ipynb.")

## Table 1: Scores and ranks per category

Each row is a model, ordered by its baseline rank. Each scenario shows the
category mean ± SE, the rank, and **ΔRank**, the signed change from the
baseline. A positive ΔRank means the model ranks higher once the task is
removed.

Below each table is a plain copy for **pasting into a Google Doc**. Select the
caption and the table in the output, copy, and paste. It keeps the bold, the
borders and the column groups. Arrows give the change in rank, and ↑ means
the model ranks higher without the task.

In [ ]:
def scenario_label(excluded: str | None) -> str:
    """Name a scenario's column group in the tables.

    Args:
        excluded: The excluded task, or None for the baseline.

    Returns:
        "All tasks" for the baseline, otherwise "w/o <task>".
    """
    return "All tasks" if excluded is None else f"w/o {TASK_DISPLAY_NAMES[excluded]}"


def format_delta(value: int) -> str:
    """Format a rank change with an explicit sign.

    Args:
        value: The signed rank change.

    Returns:
        "+n", "-n" or "0".
    """
    return f"{value:+d}" if value else "0"


def category_exclusions(category: str) -> list[str | None]:
    """List the scenarios of a category, the baseline first.

    Args:
        category: The name of the category.

    Returns:
        None for the baseline, followed by each excluded task of the category.
    """
    return [None] + [task for task in EXCLUDED_TASKS if TASK_CATEGORY[task] == category]


def scenario_rows(category: str, excluded: str | None) -> pd.DataFrame:
    """Select one scenario of a category, in baseline-rank order.

    Args:
        category: The name of the category.
        excluded: The excluded task, or None for the baseline.

    Returns:
        The scenario's rows of `ablation`, indexed by model and ordered by the
        model's baseline rank, so that every scenario lines up row for row.
    """
    selected = ablation[ablation["category"] == category]
    order = (
        selected[selected["scenario"] == BASELINE]
        .sort_values(["rank", "model"])["model"]
        .tolist()
    )
    mask = (
        selected["excluded_task"].isna()
        if excluded is None
        else selected["excluded_task"] == excluded
    )

    return selected[mask].set_index("model").loc[order]


def category_table(category: str) -> pd.DataFrame:
    """Build the wide score-and-rank table of one category.

    Args:
        category: The name of the category.

    Returns:
        One row per model in baseline-rank order, with a column group per
        scenario.
    """
    columns = {}
    for excluded in category_exclusions(category):
        rows = scenario_rows(category, excluded)
        order = rows.index.tolist()
        label = scenario_label(excluded)
        columns[(label, "Mean ± SE")] = [
            f"{mean:.3f} ± {se:.3f}" for mean, se in zip(rows["mean"], rows["se"])
        ]
        columns[(label, "Rank")] = rows["rank"].tolist()
        if excluded is not None:
            columns[(label, "ΔRank")] = rows["delta_rank"].map(format_delta).tolist()

    table = pd.DataFrame(
        columns, index=pd.Index([MODEL_DISPLAY_NAMES[m] for m in order], name="Model")
    )
    table.columns = pd.MultiIndex.from_tuples(table.columns)

    return table


def rank_with_change(rank: int, delta: int | None) -> str:
    """Format a rank, with an arrow for its change from the baseline.

    Args:
        rank: The rank.
        delta: The signed rank change, or None for the baseline.

    Returns:
        E.g. "3", "3 (↑2)", "5 (↓1)" or "4 (=)".
    """
    if delta is None:
        return str(rank)
    if delta == 0:
        return f"{rank} (=)"

    return f"{rank} ({'↑' if delta > 0 else '↓'}{abs(delta)})"


def category_cells(
    category: str,
) -> tuple[list[tuple[str, int]], list[str], list[list[str]], set[tuple[int, int]]]:
    """Lay out a category's ablation table as plain text cells.

    This is the compact layout shared by the Google Docs and Markdown tables:
    a score and a rank per scenario, with the rank change shown as an arrow.

    Args:
        category: The name of the category.

    Returns:
        The column groups as (label, width) pairs, the column headers, the
        cells of each row, and the (row, column) positions of the best score
        in each scenario.
    """
    exclusions = category_exclusions(category)
    scenarios = {excluded: scenario_rows(category, excluded) for excluded in exclusions}

    groups = [("", 1)] + [(scenario_label(excluded), 2) for excluded in exclusions]
    header = ["Model"] + ["Mean ± SE", "Rank"] * len(exclusions)
    rows = []
    bold = set()
    for i, model in enumerate(scenarios[None].index):
        cells = [MODEL_DISPLAY_NAMES[model]]
        for excluded, frame in scenarios.items():
            row = frame.loc[model]
            if model == frame["mean"].idxmax():
                bold.add((i, len(cells)))
            delta = None if excluded is None else int(row["delta_rank"])
            cells += [
                f"{row['mean']:.3f} ± {row['se']:.3f}",
                rank_with_change(int(row["rank"]), delta),
            ]
        rows.append(cells)

    return groups, header, rows, bold


CELL_STYLE = "border: 1px solid #808080; padding: 2px 6px;"


def google_docs_html(
    caption: str,
    header: list[str],
    rows: list[list[str]],
    groups: list[tuple[str, int]] | None = None,
    bold: set[tuple[int, int]] | None = None,
    n_text_columns: int = 1,
) -> HTML:
    """Render a table as plain HTML that pastes cleanly into a Google Doc.

    Every style is inline, since a pasted fragment keeps no style sheet. The
    colours are set explicitly so that a dark editor theme does not paste
    white text into the document.

    Args:
        caption: A bold line shown above the table.
        header: The column headers.
        rows: The cells of each row, as strings.
        groups: Optional column groups as (label, width) pairs, shown as a
            header row above `header`.
        bold: The (row, column) positions of the cells to embolden.
        n_text_columns: The number of leading, left-aligned text columns.

    Returns:
        The table, ready to display.
    """
    bold = bold or set()

    def cell(
        tag: str, text: str, column: int, extra: str = "", is_bold: bool = False
    ) -> str:
        if column < n_text_columns:
            align = "left"
        else:
            align = "center" if tag == "th" else "right"
        content = html.escape(text)
        if is_bold:
            content = f"<b>{content}</b>"
        return (
            f'<{tag} style="{CELL_STYLE} text-align: {align};"{extra}>{content}</{tag}>'
        )

    lines = [
        '<div style="background: #ffffff; color: #000000; padding: 8px; '
        'font-family: Arial, sans-serif; font-size: 10pt;">',
        f"<p><b>{html.escape(caption)}</b></p>",
        '<table style="border-collapse: collapse; background: #ffffff; '
        'color: #000000; font-family: Arial, sans-serif; font-size: 10pt;">',
    ]
    if groups:
        column = 0
        group_cells = []
        for label, width in groups:
            group_cells.append(cell("th", label, column, f' colspan="{width}"'))
            column += width
        lines.append("<tr>" + "".join(group_cells) + "</tr>")
    lines.append(
        "<tr>" + "".join(cell("th", text, j) for j, text in enumerate(header)) + "</tr>"
    )
    for i, row in enumerate(rows):
        lines.append(
            "<tr>"
            + "".join(
                cell("td", text, j, is_bold=(i, j) in bold)
                for j, text in enumerate(row)
            )
            + "</tr>"
        )
    lines += ["</table>", "</div>"]

    return HTML("\n".join(lines))


def highlight_moves(value: str) -> str:
    """Colour a nonzero rank change so that moves stand out.

    Args:
        value: A formatted ΔRank cell.

    Returns:
        The cell's CSS.
    """
    if value.startswith("+"):
        return "color: green; font-weight: bold"
    if value.startswith("-"):
        return "color: red; font-weight: bold"

    return ""


category_tables = {}
for category in AFFECTED_CATEGORIES:
    table = category_table(category)
    category_tables[category] = table

    delta_columns = [column for column in table.columns if column[1] == "ΔRank"]
    display(
        table.style.map(highlight_moves, subset=delta_columns).set_caption(
            f"{CATEGORIES[category].display_name}, full fine-tuning"
        )
    )

    groups, header, rows, bold = category_cells(category)
    display(
        google_docs_html(
            f"{CATEGORIES[category].display_name} category score after full "
            "fine-tuning, with each task of fewer than 500 training examples "
            "excluded in turn",
            header,
            rows,
            groups=groups,
            bold=bold,
        )
    )

## Table 2: Rank stability

This table gives one row per exclusion, comparing the ranking without that
task to the baseline ranking of its category:

* Kendall's $\tau$ and Spearman's $\rho$ measure rank agreement. 1 means the
  order is identical.
* *Models moved* counts the models whose rank changed, and *Max |ΔRank|* is
  the largest single move.
* *Top-1 kept* says whether the best model is unchanged.
* *Mean |ΔScore|* is the mean absolute change in category score.

A copy for Google Docs follows the table, as in Table 1.

In [ ]:
summary = []
for task in EXCLUDED_TASKS:
    category = TASK_CATEGORY[task]
    merged = (
        ablation[
            (ablation["category"] == category) & (ablation["scenario"] == BASELINE)
        ]
        .set_index("model")[["mean", "rank"]]
        .join(
            ablation[ablation["excluded_task"] == task].set_index("model")[
                ["mean", "rank", "delta_rank", "delta_mean"]
            ],
            lsuffix="_baseline",
        )
    )

    summary.append(
        {
            "Category": CATEGORIES[category].display_name,
            "Excluded task": TASK_DISPLAY_NAMES[task],
            "Kendall τ": kendalltau(merged["mean_baseline"], merged["mean"]).statistic,
            "Spearman ρ": spearmanr(merged["mean_baseline"], merged["mean"]).statistic,
            "Models moved": int((merged["delta_rank"] != 0).sum()),
            "Max |ΔRank|": int(merged["delta_rank"].abs().max()),
            "Top-1 kept": merged["rank_baseline"].idxmin() == merged["rank"].idxmin(),
            "Mean |ΔScore|": merged["delta_mean"].abs().mean(),
        }
    )

summary = pd.DataFrame(summary)
summary.to_csv("small_task_exclusion_ablation_summary.csv", index=False)


def summary_cells() -> tuple[list[str], list[list[str]]]:
    """Lay out the rank-stability summary as plain text cells.

    Returns:
        The column headers and the cells of each row.
    """
    header = [
        "Category",
        "Excluded task",
        "Kendall τ",
        "Spearman ρ",
        "Models moved",
        "Max |Δ| rank",
        "Top-1 kept",
        "Mean |Δ| score",
    ]
    rows = [
        [
            row["Category"],
            row["Excluded task"],
            f"{row['Kendall τ']:.3f}",
            f"{row['Spearman ρ']:.3f}",
            f"{row['Models moved']} / {len(MODELS)}",
            str(row["Max |ΔRank|"]),
            "Yes" if row["Top-1 kept"] else "No",
            f"{row['Mean |ΔScore|']:.4f}",
        ]
        for _, row in summary.iterrows()
    ]

    return header, rows


display(
    summary.style.format(
        {"Kendall τ": "{:.3f}", "Spearman ρ": "{:.3f}", "Mean |ΔScore|": "{:.4f}"}
    ).hide(axis="index")
)
display(
    google_docs_html(
        "Rank stability after full fine-tuning when each task of fewer than 500 "
        "training examples is excluded from its category",
        *summary_cells(),
        n_text_columns=2,
    )
)

## Paper tables (LaTeX)

These are booktabs tables in the style of `create_results_tables.ipynb`, with
the same siunitx setup, bold series and scaling. There is one table per
affected category, plus the rank-stability summary. Each scenario is a column
group with:

* the score as `mean(SE)`, with the best model of the scenario in bold;
* the model's rank;
* for an exclusion, the signed rank change $\Delta$ from the baseline.

The tables require `\usepackage{booktabs}`, `\usepackage{siunitx}` and
`\usepackage{graphicx}` in the document preamble.

In [ ]:
# The same siunitx setup, bold series and font size as
# `create_results_tables.ipynb`, which explains each choice. They are copied so
# that the ablation tables typeset identically to the paper's results tables.
SISETUP_OPTIONS = [
    "table-format = 1.3(3)",
    "uncertainty-mode = separate",
    "retain-zero-uncertainty = true",
    "mode = text",
    "reset-text-series = false",
]
BOLD_SERIES = r"\fontseries{b}\selectfont"
TABLE_FONT_SIZE = r"\fontsize{10}{12}\selectfont"

METRIC_NAMES = {"classification": "macro F1", "segmentation": "positive Dice"}


def category_metric(category: str) -> str:
    """Name the metric a category is scored with.

    Args:
        category: The name of the category.

    Returns:
        The metric's display name, e.g. "macro F1".
    """
    return METRIC_NAMES[get_task(CATEGORIES[category].tasks[0]).task_type]


def latex_delta(value: int) -> str:
    """Format a rank change for LaTeX, with a true minus sign.

    Args:
        value: The signed rank change.

    Returns:
        "$+n$", "$-n$" or "0".
    """
    return f"${value:+d}$" if value else "0"


def latex_score(mean: float, se: float, is_best: bool) -> str:
    """Format a mean and standard error as a siunitx table cell.

    Copied from `format_cell` in `create_results_tables.ipynb`.

    Args:
        mean: The category score.
        se: Its standard error.
        is_best: Whether this is the best score of its scenario.

    Returns:
        The cell in siunitx uncertainty notation, prefixed with the bold font
        series if it is the best. The prefix is deliberately not braced, which
        would stop siunitx aligning the cell.
    """
    cell = f"{mean:.3f}({se:.3f})"

    return f"{BOLD_SERIES} {cell}" if is_best else cell


def latex_category_frame(category: str) -> pd.DataFrame:
    r"""Build a category's ablation table as LaTeX cell strings.

    Args:
        category: The name of the category.

    Returns:
        One row per model in baseline-rank order, with a two-level column index
        of (scenario, Score / Rank / $\Delta$).
    """
    columns = {}
    for excluded in category_exclusions(category):
        rows = scenario_rows(category, excluded)
        best = rows["mean"].idxmax()
        label = scenario_label(excluded)

        columns[(label, "Score")] = [
            latex_score(row["mean"], row["se"], model == best)
            for model, row in rows.iterrows()
        ]
        columns[(label, "Rank")] = rows["rank"].astype(str).tolist()
        if excluded is not None:
            columns[(label, r"$\Delta$")] = rows["delta_rank"].map(latex_delta).tolist()

    frame = pd.DataFrame(
        columns, index=[MODEL_DISPLAY_NAMES[model] for model in rows.index]
    )
    frame.columns = pd.MultiIndex.from_tuples(frame.columns)

    return frame


def insert_cmidrules(tabular: str, columns: pd.Index) -> str:
    r"""Insert booktabs rules underneath the column group headers.

    Copied from `create_results_tables.ipynb`.

    Args:
        tabular: The LaTeX tabular environment produced by `Styler.to_latex`.
        columns: The two-level column index of the tabulated DataFrame.

    Returns:
        The tabular environment with a rule under each column group header.

    Raises:
        ValueError: If the tabular environment has no column group header row.
    """
    categories = columns.get_level_values(0)
    rules = []
    column = 2  # The first column holds the model names.
    for category in dict.fromkeys(categories):
        width = sum(category == other for other in categories)
        rules.append(rf"\cmidrule(lr){{{column}-{column + width - 1}}}")
        column += width

    lines = tabular.splitlines()
    header_lines = [i for i, line in enumerate(lines) if r"\multicolumn" in line]
    if not header_lines:
        raise ValueError("No column group header found in the tabular environment.")
    lines.insert(header_lines[0] + 1, " ".join(rules))

    return "\n".join(lines)


def wrap_table(tabular: str, caption: str, label: str, sisetup: bool) -> str:
    """Wrap a tabular environment in a scaled, captioned table float.

    Args:
        tabular: The LaTeX tabular environment.
        caption: The caption of the table.
        label: The label used to reference the table.
        sisetup: Whether to emit the siunitx setup for the score columns.

    Returns:
        The complete LaTeX source of the table.
    """
    setup = (
        [
            "\n".join(
                [r"\sisetup{", *(f"  {option}," for option in SISETUP_OPTIONS), "}"]
            )
        ]
        if sisetup
        else []
    )

    return "\n".join(
        [
            r"\begin{table}[t]",
            r"\centering",
            rf"\caption{{{caption}}}",
            rf"\label{{{label}}}",
            TABLE_FONT_SIZE,
            r"\setlength{\tabcolsep}{3pt}",
            *setup,
            r"\resizebox{\textwidth}{!}{%",
            tabular.strip(),
            r"}",
            r"\end{table}",
            "",
        ]
    )


def category_latex(category: str) -> str:
    """Render a category's ablation table as a booktabs LaTeX table.

    Args:
        category: The name of the category.

    Returns:
        The complete LaTeX source of the table.
    """
    frame = latex_category_frame(category)
    column_format = "l" + "".join(
        "S" if kind == "Score" else "c" for _, kind in frame.columns
    )
    tabular = frame.style.hide(axis="index", names=True).to_latex(
        hrules=True, siunitx=True, multicol_align="c", column_format=column_format
    )
    display_name = CATEGORIES[category].display_name

    return wrap_table(
        insert_cmidrules(tabular.strip(), frame.columns),
        caption=(
            f"The mean and standard error of the {display_name.lower()} category "
            f"score ({category_metric(category)}) after full fine-tuning, with all "
            "tasks and with each task of fewer than 500 training examples excluded "
            "in turn. $\\Delta$ is the change in rank from all tasks; a positive "
            "value means the model ranks higher without the task. The best score in "
            "each column is shown in bold."
        ),
        label=f"tab:small_task_exclusion_{category}",
        sisetup=True,
    )


def summary_latex() -> str:
    """Render the rank-stability summary as a booktabs LaTeX table.

    Returns:
        The complete LaTeX source of the table.
    """
    frame = pd.DataFrame(
        {
            "Category": summary["Category"],
            "Excluded task": summary["Excluded task"],
            r"Kendall's $\tau$": summary["Kendall τ"].map("{:.3f}".format),
            r"Spearman's $\rho$": summary["Spearman ρ"].map("{:.3f}".format),
            "Models moved": summary["Models moved"].astype(str),
            r"Max $|\Delta|$ rank": summary["Max |ΔRank|"].astype(str),
            "Top-1 kept": summary["Top-1 kept"].map({True: "Yes", False: "No"}),
            r"Mean $|\Delta|$ score": summary["Mean |ΔScore|"].map("{:.4f}".format),
        }
    )
    tabular = frame.style.hide(axis="index").to_latex(
        hrules=True, column_format="ll" + "c" * (frame.shape[1] - 2)
    )

    return wrap_table(
        tabular,
        caption=(
            "The stability of the model rankings after full fine-tuning when each "
            "task of fewer than 500 training examples is excluded from its category, "
            "compared with the ranking over all of the category's tasks."
        ),
        label="tab:small_task_exclusion_summary",
        sisetup=False,
    )


latex_tables = {
    **{category: category_latex(category) for category in AFFECTED_CATEGORIES},
    "summary": summary_latex(),
}
for name, latex in latex_tables.items():
    Path(f"small_task_exclusion_ablation_{name}.tex").write_text(latex)
    print(latex)

## Rebuttal tables (Markdown)

The same results as GitHub-flavoured Markdown tables, which OpenReview
renders. Scores are `mean ± SE`, with the best in each column in bold. Each
rank change is shown as an arrow next to the rank: ↑ means the model ranks
higher without the task, and = means no change. Unicode is used instead of
LaTeX, so the tables render without the math renderer.

Everything is written to `small_task_exclusion_ablation_rebuttal.md`, ready
to paste.

In [ ]:
def markdown_table(
    header: list[str], rows: list[list[str]], n_text_columns: int = 1
) -> str:
    """Render a GitHub-flavoured Markdown table.

    The leading text columns are left-aligned and the rest are right-aligned.
    A pipe inside a cell is escaped so that it does not end the cell.

    Args:
        header: The column headers.
        rows: The cells of each row, as strings.
        n_text_columns: The number of leading columns holding text.

    Returns:
        The Markdown source of the table.
    """
    alignments = [":---"] * n_text_columns + ["---:"] * (len(header) - n_text_columns)

    def line(cells: list[str]) -> str:
        return "| " + " | ".join(cell.replace("|", "\\|") for cell in cells) + " |"

    lines = [line(header), "| " + " | ".join(alignments) + " |"]
    lines += [line(row) for row in rows]

    return "\n".join(lines)


def category_markdown(category: str) -> str:
    """Render a category's ablation table as Markdown.

    Markdown has no column groups, so each scenario's label heads its score
    column.

    Args:
        category: The name of the category.

    Returns:
        The Markdown source of the table.
    """
    groups, _, rows, bold = category_cells(category)
    header = ["Model"]
    for label, _ in groups[1:]:
        header += [label, "Rank"]
    rows = [
        [f"**{text}**" if (i, j) in bold else text for j, text in enumerate(row)]
        for i, row in enumerate(rows)
    ]

    return markdown_table(header, rows)


def summary_markdown() -> str:
    """Render the rank-stability summary as Markdown.

    Returns:
        The Markdown source of the table.
    """
    return markdown_table(*summary_cells(), n_text_columns=2)


sections = [
    "### Effect of excluding tasks with fewer than 500 training examples",
    (
        "Each task with fewer than 500 training examples is removed from its "
        "category one at a time, and the category score of every model is "
        "recomputed with the evaluation protocol (full fine-tuning, "
        f"{len(SEEDS)} runs, {N_RESAMPLES} test-set resamples). Scores are "
        "mean ± SE, with the best in each column in bold. Arrows give the change "
        "in rank from the full category (↑ means the model ranks higher once the "
        "task is removed)."
    ),
]
for category in AFFECTED_CATEGORIES:
    sections += [
        f"**{CATEGORIES[category].display_name}** ({category_metric(category)})",
        category_markdown(category),
    ]
sections += ["**Rank stability**", summary_markdown()]

rebuttal = "\n\n".join(sections) + "\n"
Path("small_task_exclusion_ablation_rebuttal.md").write_text(rebuttal)
print(rebuttal)

## Reading the results

A rank change is meaningful only when the two models it swaps are separated
by more than their uncertainty. Many models in a category lie within one SE
of each other, and swaps among them reflect noise in both the baseline and
the ablation, not a dependence on the excluded task. A high Kendall's
$\tau$, an unchanged top model and small, local ΔRank values together
indicate that the small tasks do not drive the ranking.